<div style="
height: 100vh;
display: flex;
flex-direction: column;
justify-content: center;
align-items: center;
text-align: center;
">

<h1>Engenharia de Controle e Segurança via Jacobiano no UR10</h1>

<hr style="width: 60%;">

<p><strong>Universidade Federal de Santa Catarina – Campus Joinville</strong></p>

<p><strong>Disciplina:</strong> Robótica e Sistemas Mecatrônicos</p>

<p><strong>Autores:</strong> Gustavo Moro e Pedro Augusto Dantas Vargas</p>

<p><strong>Professor:</strong> Anelize Zomkowski Salvi</p>

<p><strong>Data:</strong> 30 de junho de 2026</p>

<hr style="width: 60%;">

</div>

<div style="page-break-after: always;"></div>

# Sumário

1. Introdução

2. Objetivos

3. Fundamentação Teórica

&emsp;3.1 O Manipulador Industrial UR10

&emsp;3.2 Cinemática Direta

&emsp;3.3 Jacobiano Geométrico

&emsp;3.4 Singularidades Cinemáticas

&emsp;3.5 Pseudo-inversa Amortecida

4. Desenvolvimento da Solução

&emsp;4.1 Arquitetura Geral do Sistema

&emsp;4.2 Comunicação com o URSim

&emsp;4.3 Monitoramento de Singularidades

&emsp;4.4 Controle Cartesiano Adaptativo

&emsp;4.5 Organização do Software

5. Resultados e Discussão

&emsp;5.1 Número de Condição

&emsp;5.2 Manipulabilidade

&emsp;5.3 Valores Singulares

&emsp;5.4 Ganho Adaptativo

&emsp;5.5 Velocidades Articulares

&emsp;5.6 Estados Operacionais

&emsp;5.7 Síntese dos Resultados

6. Conclusão

Referências
<div style="page-break-after: always;"></div>

# 1 Introdução

A utilização de robôs industriais tem crescido significativamente nos últimos anos, impulsionada pela necessidade de aumentar a produtividade, a precisão e a segurança em processos de manufatura. Manipuladores robóticos, como o UR10 da Universal Robots, são amplamente empregados em operações de montagem, soldagem, inspeção e movimentação de materiais, executando tarefas complexas com elevada repetibilidade. Entretanto, para que essas operações ocorram de forma eficiente e segura, torna-se necessário compreender e controlar o comportamento cinemático do manipulador durante sua movimentação.

Nesse contexto, o Jacobiano Geométrico constitui uma das ferramentas matemáticas mais importantes da robótica, pois estabelece a relação entre as velocidades das juntas e a velocidade do efetuador final. Além disso, sua transposta permite relacionar forças aplicadas ao efetuador com os torques desenvolvidos pelos atuadores. Essas propriedades fazem do Jacobiano um elemento fundamental para aplicações de controle, planejamento de trajetórias e monitoramento das condições operacionais do robô.

Um dos principais desafios associados ao uso do Jacobiano está relacionado às singularidades cinemáticas. Nessas configurações, o manipulador perde capacidade de movimentação em determinadas direções, tornando o sistema numericamente mal condicionado e podendo provocar movimentos instáveis, comandos excessivos às juntas e até paradas de proteção do equipamento. Por esse motivo, o monitoramento contínuo dessas condições representa uma importante estratégia para aumentar a confiabilidade e a segurança da operação.

Diante desse cenário, este trabalho apresenta o desenvolvimento de duas aplicações utilizando o robô UR10 no ambiente de simulação URSim. A primeira consiste em um sistema de monitoramento de singularidades baseado em métricas extraídas do Jacobiano Geométrico, enquanto a segunda implementa um controlador cartesiano adaptativo utilizando sua pseudo-inversa amortecida. Ambas as soluções foram desenvolvidas em Python com comunicação via RTDE e validadas por meio da análise dos dados obtidos durante a simulação, demonstrando a aplicação prática dos conceitos estudados na disciplina.
<div style="page-break-after: always;"></div>
<br>

<div style="text-align: center; margin: 20px 0;">

  <div style="text-align: center;"><b>Figura 1 – Manipulador UR10 e seus seis graus de liberdade.</b></div>

  <img src="figuras/UR10-robot.png"
       width="250"
       style="display:block; margin: 10px auto;">

  <div style="text-align: center;">Fonte: RoboDK (2026).</div>

</div>

<br>


# 2 Objetivos

## 2.1 Objetivo Geral

Desenvolver e validar duas aplicações baseadas no Jacobiano Geométrico do manipulador industrial UR10, utilizando o simulador URSim e comunicação via RTDE, demonstrando a aplicação prática dos conceitos de cinemática diferencial no monitoramento de singularidades e no controle adaptativo do robô.

## 2.2 Objetivos Específicos

Para atingir o objetivo geral, foram definidos os seguintes objetivos específicos:

- Implementar o modelo cinemático do manipulador UR10 em Python;
- Calcular o Jacobiano Geométrico a partir da configuração instantânea das juntas;
- Desenvolver um algoritmo capaz de monitorar continuamente o número de condição, a manipulabilidade e os valores singulares do Jacobiano;
- Implementar um sistema de proteção que adapte o comportamento do robô conforme o risco de aproximação de singularidades;
- Desenvolver um controlador cartesiano baseado na pseudo-inversa amortecida do Jacobiano;
- Integrar os algoritmos ao ambiente URSim utilizando comunicação RTDE;
- Registrar os dados da simulação e analisar o desempenho do sistema por meio de gráficos e métricas obtidas durante os testes.

# 3 Fundamentação Teórica

## 3.1 O Manipulador Industrial UR10

O UR10 é um manipulador robótico colaborativo de seis graus de liberdade desenvolvido pela Universal Robots para aplicações industriais que exigem precisão, flexibilidade e facilidade de programação. Seu projeto mecânico permite executar tarefas como montagem, inspeção, soldagem, paletização e movimentação de materiais, tornando-o amplamente empregado em linhas de produção automatizadas.

Por possuir uma arquitetura serial composta por seis juntas rotativas, o UR10 apresenta elevada capacidade de posicionamento e orientação do efetuador final dentro de seu espaço de trabalho. Entretanto, diferentes configurações articulares podem produzir a mesma posição cartesiana do efetuador, tornando necessária a utilização de modelos matemáticos para descrever seu comportamento cinemático.

Neste trabalho, o manipulador foi utilizado no ambiente de simulação URSim, permitindo implementar e validar algoritmos de monitoramento e controle sem a necessidade de acesso ao robô físico. As posições articulares fornecidas pelo simulador serviram como base para o cálculo contínuo do Jacobiano Geométrico e para a execução das aplicações desenvolvidas.

<br>

<div style="text-align: center; margin: 20px 0;">

  <div style="text-align: center;"><b>Figura 2 – Estrutura cinemática e identificação das juntas do manipulador UR10.</b></div>

  <img src="figuras/UR10_juntas.png"
       width="250"
       style="display:block; margin: 10px auto;">

  <div style="text-align: center;">Fonte: Adaptado de Williams (2024).</div>

</div>

<br>
<div style="page-break-after: always;"></div>

## 3.2 Cinemática Direta

A cinemática direta corresponde ao conjunto de equações responsáveis por determinar a posição e a orientação do efetuador final a partir das variáveis articulares do manipulador. Para isso, é comum utilizar a convenção de Denavit-Hartenberg (DH), que descreve cada elo do robô por meio de quatro parâmetros geométricos padronizados.

Neste projeto, os parâmetros DH do UR10 foram implementados em Python para calcular as matrizes de transformação homogênea entre os elos consecutivos. A multiplicação sucessiva dessas matrizes permite determinar a pose completa do efetuador final em relação ao sistema de coordenadas da base do robô.

A transformação homogênea utilizada é representada pela Equação (1).

$$
{}^{i-1}T_i=
\begin{bmatrix}
R & p\\
0 & 1
\end{bmatrix}
\tag{1}
$$

em que $R$ representa a matriz de rotação do elo e $p$ corresponde ao vetor posição em relação ao sistema de coordenadas anterior.

O resultado da cinemática direta constitui a base para o cálculo do Jacobiano Geométrico, utilizado nas aplicações desenvolvidas neste trabalho.

## 3.3 Jacobiano Geométrico

O Jacobiano Geométrico estabelece a relação entre as velocidades articulares do manipulador e a velocidade cartesiana do efetuador final. Essa relação permite determinar como pequenas variações nos ângulos das juntas influenciam diretamente o movimento do robô no espaço operacional.

A relação entre velocidades é dada pela Equação (2).

$$
\dot{x}=J(q)\dot{q}
\tag{2}
$$

onde:

- $\dot{x}$ representa a velocidade cartesiana do efetuador final;
- $J(q)$ representa o Jacobiano Geométrico;
- $\dot{q}$ representa o vetor de velocidades articulares.

Além do mapeamento entre velocidades, a transposta do Jacobiano permite relacionar forças aplicadas ao efetuador final com os torques desenvolvidos nas juntas do manipulador, conforme mostrado na Equação (3).

$$
\tau = J^T(q)F
\tag{3}
$$

Essa característica faz do Jacobiano uma das ferramentas mais importantes da robótica, sendo amplamente empregado em problemas de controle, planejamento de trajetórias, estimação de esforços mecânicos e monitoramento de singularidades.

## 3.4 Singularidades Cinemáticas

As singularidades cinemáticas correspondem a configurações nas quais o manipulador perde capacidade de movimentação em determinadas direções do espaço cartesiano. Nessas situações, o Jacobiano torna-se mal condicionado ou perde posto, fazendo com que pequenas velocidades desejadas possam exigir velocidades articulares excessivamente elevadas.

<br>

<div style="text-align: center; margin: 20px 0;">

  <div style="text-align: center;"><b>Figura 3 – Representação da manipulabilidade em diferentes configurações do manipulador.</b></div>

  <img src="figuras/ur3-singularities.png"
       width="350"
       style="display:block; margin: 10px auto;">

  <div style="text-align: center;">Fonte: Retirado de Mecademic (2026).</div>

</div>

<br>

Para monitorar essas condições, este trabalho empregou três métricas principais: o número de condição, o índice de manipulabilidade e os valores singulares do Jacobiano.

O número de condição é calculado pela razão entre o maior e o menor valor singular da matriz, conforme apresentado na Equação (4).

$$
\kappa(J)=\frac{\sigma_{max}}{\sigma_{min}}
\tag{4}
$$

Quanto maior o número de condição, maior a proximidade de uma configuração singular.

Outra métrica utilizada foi o índice de manipulabilidade proposto por Yoshikawa, dado pela Equação (5).

$$
w(q)=\sqrt{\det(JJ^T)}
\tag{5}
$$

Valores reduzidos dessa métrica indicam diminuição da capacidade do manipulador em produzir movimentos em diferentes direções do espaço cartesiano, caracterizando regiões de pior desempenho cinemático.
<div style="page-break-after: always;"></div>

## 3.5 Pseudo-inversa Amortecida

Em aplicações de controle cartesiano, a pseudo-inversa do Jacobiano é utilizada para converter velocidades desejadas do efetuador final em velocidades articulares equivalentes. Entretanto, quando o manipulador se aproxima de uma singularidade, a pseudo-inversa convencional pode produzir comandos excessivos devido ao elevado número de condição da matriz.

Para minimizar esse problema, foi empregada neste trabalho a pseudo-inversa amortecida (*Damped Least Squares*), representada pela Equação (6).

$$
J^{+}=J^{T}(JJ^{T}+\lambda^{2}I)^{-1}
\tag{6}
$$

em que $\lambda$ representa o fator de amortecimento responsável por melhorar a estabilidade numérica do controlador.

Além da utilização dessa técnica, foi implementado um ganho adaptativo que reduz automaticamente a intensidade dos comandos enviados ao manipulador conforme o número de condição aumenta. Dessa forma, o controlador preserva movimentos suaves mesmo durante a aproximação de regiões de pior condicionamento, contribuindo para a segurança e estabilidade da operação.

# 4 Desenvolvimento da Solução

## 4.1 Arquitetura Geral do Sistema

O sistema desenvolvido foi estruturado de forma modular, permitindo separar as atividades de comunicação com o simulador, processamento matemático, controle do manipulador e registro dos dados obtidos durante a simulação. Essa organização reduz o acoplamento entre os módulos, facilita a manutenção do código e possibilita futuras expansões sem a necessidade de alterações significativas na estrutura principal do software.

Durante cada ciclo de execução, o algoritmo realiza inicialmente a leitura das posições articulares do manipulador por meio da interface RTDE. Em seguida, essas informações são utilizadas para calcular a cinemática do robô e o respectivo Jacobiano Geométrico. A partir desse momento, os dados passam a alimentar simultaneamente o módulo responsável pelo monitoramento das singularidades e o controlador cartesiano adaptativo.

Por fim, todas as variáveis de interesse são armazenadas em arquivo CSV para posterior geração dos gráficos e análise dos resultados experimentais. Esse fluxo permite acompanhar continuamente o comportamento do sistema durante toda a execução da trajetória.

<div style="page-break-after: always;"></div>
<div style="text-align: center; margin: 20px 0;">

  <div style="text-align: center;"><b>Figura 4 – Arquitetura geral do sistema desenvolvido.</b></div>

  <img src="figuras/arquitetura.png"
       width="600"
       style="display:block; margin: 10px auto;">

  <div style="text-align: center;">Fonte: Elaborado pelo autor (2026).</div>

</div>

<br>

## 4.2 Comunicação com o URSim

A integração entre o algoritmo desenvolvido e o manipulador foi realizada utilizando a interface **RTDE (Real-Time Data Exchange)**, disponibilizada pela Universal Robots para comunicação em tempo real com o controlador do robô. Essa interface permite obter continuamente as posições das juntas e demais informações necessárias para aplicações de monitoramento e controle.

Durante a execução do programa, as posições articulares são adquiridas a cada ciclo de atualização e utilizadas para calcular o Jacobiano Geométrico correspondente à configuração instantânea do manipulador. Com base nessas informações, o sistema determina o estado operacional do robô e calcula os comandos de velocidade empregados pelo controlador adaptativo.

Além da integração com o URSim, foi implementado um modo de simulação local, utilizado para validar o funcionamento dos algoritmos mesmo na ausência do simulador. Nesse modo, trajetórias articulares sintéticas são geradas automaticamente, permitindo verificar toda a lógica de monitoramento, controle e registro dos dados antes da execução no ambiente de simulação.
<div style="page-break-after: always;"></div>

## 4.3 Monitoramento de Singularidades

A primeira aplicação desenvolvida neste trabalho consiste em um sistema de monitoramento contínuo das condições cinemáticas do manipulador. Em cada ciclo de execução, o Jacobiano Geométrico é calculado e utilizado para determinar o número de condição, o índice de manipulabilidade e os valores singulares da matriz.

Essas métricas são utilizadas para classificar automaticamente o estado operacional do manipulador em quatro níveis: **NORMAL**, **WARNING**, **RISK** e **CRITICAL**. Essa classificação permite identificar a aproximação de regiões potencialmente singulares antes que ocorram perdas significativas de desempenho ou o acionamento dos mecanismos internos de proteção do robô.

Além da classificação dos estados, o algoritmo associa a cada condição um fator de redução da velocidade de operação. Dessa forma, quanto mais próximo o manipulador estiver de uma singularidade, menor será a velocidade permitida para execução da trajetória. Caso seja identificada uma condição classificada como **CRITICAL**, o sistema interrompe preventivamente o movimento, preservando a estabilidade e a segurança da operação.

## 4.4 Controle Cartesiano Adaptativo

A segunda aplicação desenvolvida utiliza o Jacobiano Geométrico para converter velocidades cartesianas desejadas em velocidades articulares, permitindo controlar o movimento do efetuador final de acordo com a configuração instantânea do manipulador.

Para aumentar a robustez do controlador, foi empregada a pseudo-inversa amortecida do Jacobiano, reduzindo a sensibilidade do sistema em regiões próximas às singularidades. Além disso, foi implementado um ganho adaptativo que varia automaticamente conforme o número de condição da matriz.

Em configurações bem condicionadas, o controlador opera com ganho máximo, permitindo maior velocidade de execução. À medida que o manipulador se aproxima de regiões de pior condicionamento, esse ganho é reduzido gradualmente, diminuindo a intensidade dos comandos enviados às juntas e preservando a estabilidade do sistema durante toda a trajetória.

## 4.5 Organização do Software

O projeto foi desenvolvido em Python utilizando uma arquitetura modular, na qual cada arquivo possui uma responsabilidade específica dentro do sistema. Essa organização facilita a manutenção do código, melhora sua legibilidade e permite que cada componente seja validado individualmente antes da integração completa da aplicação.

A Tabela 1 apresenta a função desempenhada por cada módulo desenvolvido.

| Arquivo | Função |
|:---------|:-------|
| `main.py` | Coordena a execução do sistema e integra todos os módulos. |
| `ur10_kinematics.py` | Implementa a cinemática direta e o cálculo do Jacobiano Geométrico. |
| `jacobian_tools.py` | Calcula número de condição, manipulabilidade, valores singulares e pseudo-inversa. |
| `singularity_monitor.py` | Monitora as condições cinemáticas e classifica o estado operacional do manipulador. |
| `adaptive_controller.py` | Calcula as velocidades articulares utilizando ganho adaptativo. |
| `data_logger.py` | Registra todas as variáveis da simulação em arquivo CSV. |
| `plot_results.py` | Gera automaticamente os gráficos utilizados na análise dos resultados. |
| `config.py` | Centraliza os parâmetros utilizados durante a execução do sistema. |

Essa divisão permitiu desenvolver cada módulo de forma independente, reduzindo a complexidade da implementação e facilitando tanto os testes individuais quanto a integração final do software.

# 5 Resultados e Discussão

Após o desenvolvimento e a implementação dos algoritmos propostos, foram realizados testes utilizando o ambiente de simulação URSim, permitindo avaliar o comportamento do sistema diante de diferentes condições cinemáticas do manipulador. Durante a execução, foram registradas variáveis relacionadas ao Jacobiano Geométrico, ao controlador adaptativo e ao estado operacional do robô, possibilitando analisar quantitativamente a resposta do sistema ao longo da trajetória executada.

Os resultados apresentados nas seções seguintes demonstram o funcionamento conjunto do monitor de singularidades e do controlador adaptativo, evidenciando como as propriedades matemáticas do Jacobiano podem ser empregadas para aumentar a estabilidade e a segurança operacional do manipulador. Cada métrica será analisada individualmente, permitindo compreender sua influência sobre o comportamento do sistema durante toda a simulação.

## 5.1 Número de Condição

Para iniciar a análise dos resultados, foi avaliada a evolução do número de condição do Jacobiano ao longo da trajetória executada. Essa métrica representa o grau de condicionamento da matriz e constitui um dos principais indicadores da proximidade de configurações singulares.

<div style="page-break-after: always;"></div>

<div style="text-align: center; margin: 20px 0;">

  <div style="text-align: center;"><b>Figura 5 – Evolução do número de condição durante a simulação.</b></div>

  <img src="figuras/numero_condicao.png"
       width="500"
       style="display:block; margin: 10px auto;">

  <div style="text-align: center;">Fonte: Dados da simulação elaborados pelo autor (2026).</div>

</div>

<br>

Observa-se que o número de condição apresentou crescimento gradual durante a simulação, variando aproximadamente entre **7,5** e **10,0**. Esse comportamento demonstra que a trajetória conduz o manipulador para regiões de menor condicionamento, permitindo validar o funcionamento do sistema de monitoramento implementado.

À medida que essa métrica aumenta, pequenas variações na velocidade cartesiana passam a exigir velocidades articulares progressivamente maiores. Por esse motivo, o número de condição foi utilizado como principal parâmetro para adaptação do controlador desenvolvido, permitindo reduzir preventivamente a velocidade do robô antes da ocorrência de situações críticas.

## 5.2 Manipulabilidade

Após analisar a evolução do número de condição, passa-se à avaliação do índice de manipulabilidade de Yoshikawa. Essa métrica complementa a análise anterior ao indicar a capacidade do manipulador em produzir movimentos em diferentes direções do espaço cartesiano.
<div style="page-break-after: always;"></div>
<br>
<div style="text-align: center; margin: 20px 0;">

  <div style="text-align: center;"><b>Figura 6 – Evolução do índice de manipulabilidade durante a simulação.</b></div>

  <img src="figuras/manipulabilidade.png"
       width="500"
       style="display:block; margin: 10px auto;">

  <div style="text-align: center;">Fonte: Dados da simulação elaborados pelo autor (2026).</div>

</div>
<br>

Os resultados mostram uma redução gradual da manipulabilidade ao longo da trajetória, variando aproximadamente entre **0,236** e **0,202**. Esse comportamento confirma a relação esperada entre essa métrica e o número de condição apresentado anteriormente.

Quanto menor o índice de manipulabilidade, menor é a capacidade do manipulador de produzir movimentos isotrópicos no espaço cartesiano. Dessa forma, observa-se que a aproximação de regiões menos favoráveis da área de trabalho resulta diretamente na diminuição dessa métrica, validando o comportamento esperado do modelo matemático implementado.

## 5.3 Valores Singulares

Em seguida, foram analisados os valores singulares do Jacobiano, responsáveis por fornecer uma descrição detalhada do condicionamento da matriz durante toda a simulação.
<div style="page-break-after: always;"></div>
<div style="text-align: center; margin: 20px 0;">

  <div style="text-align: center;"><b>Figura 7 – Evolução dos valores singulares do Jacobiano.</b></div>

  <img src="figuras/valores_singulares.png"
       width="500"
       style="display:block; margin: 10px auto;">

  <div style="text-align: center;">Fonte: Dados da simulação elaborados pelo autor (2026).</div>

</div>
<br>
Observa-se que o maior valor singular permaneceu praticamente constante durante toda a execução, enquanto o menor valor singular apresentou redução gradual conforme o manipulador se aproximava de regiões de pior condicionamento.

Como o número de condição corresponde à razão entre esses dois valores, o comportamento observado explica diretamente o aumento apresentado na Figura 5. Esses resultados confirmam a consistência matemática dos cálculos realizados pelo algoritmo e demonstram que as alterações verificadas nas demais métricas decorrem efetivamente das mudanças na configuração cinemática do manipulador.

## 5.4 Ganho Adaptativo

Compreendido o comportamento do Jacobiano, passa-se à análise da resposta do controlador desenvolvido. Nesta etapa, observa-se a variação do ganho adaptativo em função do número de condição calculado durante a simulação.
<br>
<div style="text-align: center; margin: 20px 0;">

  <div style="text-align: center;"><b>Figura 8 – Evolução do ganho adaptativo aplicado pelo controlador.</b></div>

  <img src="figuras/ganho_adaptativo.png"
       width="500"
       style="display:block; margin: 10px auto;">

  <div style="text-align: center;">Fonte: Dados da simulação elaborados pelo autor (2026).</div>

</div>

<br>

O gráfico mostra que o controlador reduziu automaticamente o ganho de operação conforme o número de condição aumentava. Inicialmente, o sistema operou com ganho máximo, reduzindo-o progressivamente ao identificar configurações menos favoráveis do manipulador.

Essa estratégia permitiu limitar os comandos enviados às juntas sem interromper imediatamente a execução da trajetória. Dessa forma, o controlador manteve a estabilidade do sistema mesmo durante a aproximação de regiões de pior condicionamento, evidenciando a eficácia da estratégia proposta.

## 5.5 Velocidades Articulares

Após avaliar o comportamento do controlador adaptativo, analisam-se as velocidades articulares calculadas para cada uma das seis juntas do manipulador. Essa etapa permite verificar como a adaptação do ganho influenciou diretamente os comandos enviados ao robô durante a execução da trajetória.
<br>
<div style="text-align: center; margin: 20px 0;">

  <div style="text-align: center;"><b>Figura 9 – Velocidades articulares calculadas pelo controlador durante a simulação.</b></div>

  <img src="figuras/velocidades_articulares.png"
       width="500"
       style="display:block; margin: 10px auto;">

  <div style="text-align: center;">Fonte: Dados da simulação elaborados pelo autor (2026).</div>

</div>
<br>

Observa-se que as velocidades articulares permaneceram contínuas durante toda a simulação, sem oscilações abruptas que pudessem comprometer a estabilidade do sistema. Embora cada junta apresente comportamento distinto, em função de sua contribuição para o movimento cartesiano do efetuador, todas responderam de maneira consistente às alterações impostas pelo controlador.

Também é possível perceber que, à medida que o ganho adaptativo foi reduzido, houve diminuição proporcional da magnitude das velocidades articulares. Esse comportamento demonstra que a estratégia proposta atuou corretamente, limitando a intensidade dos comandos em configurações menos favoráveis do manipulador e reduzindo a possibilidade de movimentos excessivos próximos às singularidades.

## 5.6 Estados Operacionais

Por fim, apresenta-se a distribuição dos estados operacionais identificados pelo algoritmo de monitoramento ao longo da simulação. Essa análise resume o comportamento global do sistema, demonstrando como o manipulador foi classificado durante toda a trajetória executada.
<br>
<div style="text-align: center; margin: 20px 0;">

  <div style="text-align: center;"><b>Figura 10 – Distribuição dos estados operacionais identificados durante a simulação.</b></div>

  <img src="figuras/distribuicao_status.png"
       width="450"
       style="display:block; margin: 10px auto;">

  <div style="text-align: center;">Fonte: Dados da simulação elaborados pelo autor (2026).</div>

</div>

<br>

Os resultados mostram que a maior parte da simulação ocorreu nos estados **WARNING** e **NORMAL**, indicando que o manipulador permaneceu predominantemente em condições seguras de operação. Entretanto, também foram registrados estados classificados como **RISK**, evidenciando que a trajetória realmente conduziu o robô para regiões de menor condicionamento, conforme planejado para validação do algoritmo.

Destaca-se ainda a ocorrência de um estado **CRITICAL**, demonstrando que o sistema foi capaz de identificar automaticamente uma condição limite e acionar sua lógica de proteção. Embora esse evento tenha ocorrido durante um intervalo reduzido, sua detecção confirma que os critérios adotados para classificação dos estados operacionais foram capazes de distinguir diferentes níveis de risco de forma consistente.

## 5.7 Síntese dos Resultados

Os resultados obtidos demonstram que as duas aplicações propostas neste trabalho atingiram os objetivos estabelecidos. O sistema desenvolvido foi capaz de calcular continuamente o Jacobiano Geométrico do manipulador, monitorar suas principais propriedades matemáticas e adaptar automaticamente o comportamento do controlador conforme as condições cinemáticas observadas durante a execução da trajetória.

A análise conjunta das métricas evidenciou uma relação coerente entre o aumento do número de condição, a redução da manipulabilidade e a diminuição do menor valor singular do Jacobiano. Essas alterações refletiram diretamente na atuação do ganho adaptativo, que reduziu progressivamente a intensidade dos comandos enviados às juntas do manipulador, contribuindo para um comportamento mais estável do sistema.

Além de validar os conceitos teóricos apresentados na fundamentação deste trabalho, os resultados demonstram que o Jacobiano Geométrico pode ser utilizado como elemento central no desenvolvimento de estratégias de controle e monitoramento para manipuladores industriais. A integração entre monitoramento de singularidades, adaptação do controlador e análise dos indicadores cinemáticos mostrou-se eficiente durante toda a simulação, evidenciando a aplicabilidade prática das técnicas estudadas na disciplina.

# 6 Conclusão

O presente trabalho teve como objetivo desenvolver aplicações práticas baseadas no Jacobiano Geométrico do manipulador UR10, explorando suas propriedades matemáticas para aumentar a segurança e a qualidade do controle durante a operação do robô. Para isso, foram implementados dois algoritmos complementares: um sistema de monitoramento de singularidades e um controlador cartesiano com ganho adaptativo, ambos integrados ao ambiente de simulação URSim.

Os resultados obtidos demonstraram que o monitoramento contínuo do número de condição, do índice de manipulabilidade e dos valores singulares permite identificar alterações relevantes nas condições cinemáticas do manipulador ao longo da trajetória. A partir dessas informações, o controlador adaptativo foi capaz de reduzir automaticamente os comandos enviados às juntas em regiões de pior condicionamento, contribuindo para um comportamento mais estável e seguro.

Além de atender aos requisitos propostos para o trabalho, o projeto permitiu aplicar, de forma integrada, conceitos de cinemática, álgebra linear, controle e programação em robótica. Dessa maneira, foi possível compreender a importância do Jacobiano Geométrico não apenas como uma ferramenta matemática, mas como um elemento fundamental para o desenvolvimento de soluções reais de monitoramento e controle em manipuladores industriais.
<div style="page-break-after: always;"></div>

# Referências

CRAIG, John J. **Introduction to Robotics: Mechanics and Control**. 4. ed. Upper Saddle River: Pearson, 2018.

SICILIANO, Bruno; SCIAVICCO, Luigi; VILLANI, Luigi; ORIOLO, Giuseppe. **Robotics: Modelling, Planning and Control**. London: Springer, 2010.

YOSHIKAWA, Tsuneo. Manipulability of robotic mechanisms. *The International Journal of Robotics Research*, v. 4, n. 2, p. 3–9, 1985.

UNIVERSAL ROBOTS. **UR10e User Manual**. Odense: Universal Robots A/S, 2025.

ROBODK. **Robot Library – Universal Robots UR10**. Disponível em: <https://robodk.com/robot/Universal-Robots/UR10>. Acesso em: 30 jun. 2026.

WILLIAMS, Robert L. **Universal Robot URe-Series Cobot Kinematics & Dynamics**. Ohio University. Disponível em: <https://people.ohio.edu/williams/html/PDF/UniversalRobotKinematics.pdf>. Acesso em: 30 jun. 2026.

MECADEMIC. **What Are Singularities in a 6-Axis Robot Arm?** Disponível em: <https://mecademic.com/insights/academic-tutorials/what-are-singularities-6-axis-robot-arm/>. Acesso em: 30 jun. 2026.